# Museum Visitors Analysis


In [1]:
import os
import sys

# Add src/ to path when running in Jupyter container (museums-data volume is /home/jovyan/data)
# For local dev: adjust path as needed
src_path = os.path.join(os.path.dirname(os.getcwd()), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

DATABASE_URL = os.environ.get("DATABASE_URL", "sqlite:////home/jovyan/data/museums.db")
MODEL_PATH = os.environ.get("MODEL_PATH", "/home/jovyan/data/models/regression.pkl")
os.environ.setdefault("MODEL_PATH", MODEL_PATH)


'/home/jovyan/data/models/regression.pkl'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

import museums.db as db
import museums.regression as regression


In [ ]:
engine = create_engine(DATABASE_URL)
db.init_db(DATABASE_URL)
session = sessionmaker(engine)()

rows = db.get_training_rows(session)
populations = np.array([r.population for r in rows])
visitors = np.array([r.visitors_annual for r in rows])

print(f"Loaded {len(rows)} museum-city pairs")


In [ ]:
regression.load_model()
print("Model loaded")


In [ ]:
from sklearn.metrics import r2_score
preds = [regression.predict(int(p)) for p in populations]
r2 = r2_score(visitors, preds)

pop_range = np.linspace(populations.min(), populations.max(), 200)
fit_line = [regression.predict(int(p)) for p in pop_range]

plt.figure(figsize=(10, 6))
plt.scatter(populations, visitors, alpha=0.7, label="Observed")
plt.plot(pop_range, fit_line, color="red", linewidth=2, label=f"Fit (R\u00b2={r2:.4f})")
plt.xlabel("City Population")
plt.ylabel("Annual Visitors")
plt.title("Museum Visitors vs City Population")
plt.legend()
plt.tight_layout()
plt.show()
